In [1]:
import pandas as pd
import numpy as np
import os
import json
import requests

from pyjstat import pyjstat
from collections import OrderedDict

c:\Users\jonas\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
dir_out = "../parsed_data/"

In [3]:
map_country_ISO = {
    "Belgium" : "BE",
    "Bulgaria" : "BG",
    "Czechia" : "CZ",
    "Denmark" : "DK",
    "Germany" : "DE",
    "Estonia" : "EE",
    "Ireland" : "IE",
    "Greece" : "EL",   ## NEW
    "Spain" : "ES",
    "France" : "FR",
    "Croatia" : "HR",
    "Italy" : "IT",
    "Cyprus" : "CY",
    "Latvia" : "LV",
    "Lithuania" : "LT",
    "Luxembourg" : "LU",
    "Hungary" : "HU",
    "Malta" : "MT",
    "Netherlands" : "NL",
    "Austria" : "AT",
    "Poland" : "PL",
    "Portugal" : "PT",
    "Romania" : "RO",
    "Slovenia" : "SI",
    "Slovakia" : "SK",
    "Finland" : "FI",
    "Sweden" : "SE",
    "United Kingdom" : "UK",  ## NEW
    "Iceland" : "IS",
    "Norway" : "NO",
    "Montenegro" : "ME",
    "North Macedonia" : "MK",
    "Albania" : "AL",
    "Serbia" : "RS",
    "Türkiye" : "TR",                     # Leon edit, corrected country name as used in the dataset 
    "Bosnia and Herzegovina" : "BA",
    "Kosovo (under United Nations Security Council Resolution 1244/99)" : "XK",
    "Moldova" : "MD",
    "Ukraine" : "UA",
    "Georgia" : "GE",
    "Liechtenstein" : "LI"                # Leon edit, added Liechtenstein 
}

In [4]:
# Eurostat queries based on query builder: 
# https://ec.europa.eu/eurostat/web/query-builder/tool
indicator = 'nrg_pc_203'
dataformat = 'JSON'

params = dict(
    sinceTimePeriod = '2020-S1',
    geo = {'AT', 'BE', 'BG', 'CY', 'CZ', 'DE', 'DK', 'EE', 'EL', 'ES', 'FI', 'FR', 'HR', 'HU', 'IE', 'IT', 'LT', 'LU', 'LV', 'MD', 'MK', 'MT', 'NL', 'NO', 'PL', 'PT', 'RO', 'RS', 'SE', 'SI', 'SK', 'TR', 'UA', 'UK', 'AL', 'BA', 'LI', 'IS', 'GE', 'ME', 'XK'},
    unit = 'KWH',
    product = '4100',
    nrg_cons='TOT_GJ',
    tax = 'X_VAT',
    currency = 'EUR',
    lang = 'en'
)

In [5]:
url = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/"+indicator+"?format="+dataformat
r = requests.get(url=url, params=params)
nrg_pc_203_temp = pd.DataFrame(pyjstat.from_json_stat(r.json(object_pairs_hook=OrderedDict))[0])

In [6]:
nrg_pc_203_temp.head()

,Time frequency,Products,Energy consumption,Unit of measure,Taxes,Currency,Geopolitical entity (reporting),Time,value
0,"Half-yearly, semesterly",Natural gas,Consumption of GJ - all bands,Kilowatt-hour,Excluding VAT and other recoverable taxes and ...,Euro,Belgium,2020-S1,NaN
1,"Half-yearly, semesterly",Natural gas,Consumption of GJ - all bands,Kilowatt-hour,Excluding VAT and other recoverable taxes and ...,Euro,Belgium,2020-S2,NaN
2,"Half-yearly, semesterly",Natural gas,Consumption of GJ - all bands,Kilowatt-hour,Excluding VAT and other recoverable taxes and ...,Euro,Belgium,2021-S1,NaN
3,"Half-yearly, semesterly",Natural gas,Consumption of GJ - all bands,Kilowatt-hour,Excluding VAT and other recoverable taxes and ...,Euro,Belgium,2021-S2,0.0471
4,"Half-yearly, semesterly",Natural gas,Consumption of GJ - all bands,Kilowatt-hour,Excluding VAT and other recoverable taxes and ...,Euro,Belgium,2022-S1,0.0676


In [7]:
#rename countries and convert to MWh
nrg_pc_203 = nrg_pc_203_temp.copy().dropna()
nrg_pc_203 = nrg_pc_203.rename(columns={'Geopolitical entity (reporting)':'country'})
nrg_pc_203['country'] = nrg_pc_203['country'].map(map_country_ISO)
nrg_pc_203['year'] = nrg_pc_203['Time'].str[:4]
nrg_pc_203 = nrg_pc_203[['country', 'year', 'value']].dropna() #need to check where the NaNs come from
nrg_pc_203 = nrg_pc_203.groupby(['country','year']).mean()
nrg_pc_203['EUR_per_MWh'] = nrg_pc_203['value'] * 1000
nrg_pc_203 = nrg_pc_203[['EUR_per_MWh']]
nrg_pc_203.head()

EUR_per_MWh
country year             
AT      2021        48.70
        2022        87.35
        2023        63.75
        2024        51.85
BA      2021        38.70

In [8]:
nrg_pc_203.reset_index().country.unique()

array(['AT', 'BA', 'BE', 'BG', 'CZ', 'DE', 'EE', 'EL', 'ES', 'FI', 'FR',
       'HR', 'HU', 'IT', 'MK', 'NL', 'PL', 'PT', 'RO', 'RS', 'SK', 'TR'],
      dtype=object)

In [9]:
nrg_pc_203.to_csv(dir_out+'price_gas_yearly_Eurostat.csv', encoding="utf-8")